## Download Script for UVA Soil Moisture (Weekly)

In [1]:
# Establish Directories
output_dir = '/global/scratch/users/cgerlein/fc_ecohydrology_scratch/CYGNSS/Data/sls_SOILMOISTURE_UVA'

In [2]:
# Import packages
#%pip install rioxarray
import urllib.request
import pandas as pd
import xarray as xr
import requests
import os

#### Get a list of the urls from the UVA dataset

In [3]:
# dataset DOI
dataset_doi = "doi:10.18130/V3/S6BXO3"
api_url = "https://dataverse.lib.virginia.edu/api/datasets/:persistentId/"
params = {"persistentId": dataset_doi}

# request dataset metadata
response = requests.get(api_url, params=params)
data = response.json()

# extract file IDs
files = data["data"]["latestVersion"]["files"]
url_ids = [file["dataFile"]["id"] for file in files]
file_names = [file["label"] for file in files]
print(f"Found {len(url_ids)} files.")

# Create list of urls
url_dir = 'https://dataverse.lib.virginia.edu/api/access/datafile'
download_dict = {
    file["label"]: f"{url_dir}/{file['dataFile']['id']}"
    for file in files
}

print('these are the links and names:')
list(download_dict.items())[:5]

Found 507 files.
these are the links and names:


[('smap_sm_400m_2015_week_01.tif',
  'https://dataverse.lib.virginia.edu/api/access/datafile/102793'),
 ('smap_sm_400m_2015_week_02.tif',
  'https://dataverse.lib.virginia.edu/api/access/datafile/102798'),
 ('smap_sm_400m_2015_week_03.tif',
  'https://dataverse.lib.virginia.edu/api/access/datafile/102795'),
 ('smap_sm_400m_2015_week_04.tif',
  'https://dataverse.lib.virginia.edu/api/access/datafile/102778'),
 ('smap_sm_400m_2015_week_05.tif',
  'https://dataverse.lib.virginia.edu/api/access/datafile/102763')]

In [4]:
# Loop
i = 0
for filename, url in list(download_dict.items())[170:]: # ADJUST FOR ALL
    output_fn = f'{output_dir}/{filename}'

    # skip if exist
    if os.path.exists(output_fn):
        print(f'[{i}] Skipping (already exists): {output_fn}')
        print('-----------')
        i += 1
        continue

    # download if not
    print(f'{[i]}Downloading: {output_fn}....')
    try:
        urllib.request.urlretrieve(url, output_fn)
        print(f'      Complete!\n-----------')
    except Exception as e:
        print(f'      Download failed ({e}). Skipping...\n-----------')
    i = i+1 

[0] Skipping (already exists): /global/scratch/users/cgerlein/fc_ecohydrology_scratch/CYGNSS/Data/sls_SOILMOISTURE_UVA/smap_sm_400m_2018_week_28.tif
-----------
[1] Skipping (already exists): /global/scratch/users/cgerlein/fc_ecohydrology_scratch/CYGNSS/Data/sls_SOILMOISTURE_UVA/smap_sm_400m_2018_week_29.tif
-----------
[2] Skipping (already exists): /global/scratch/users/cgerlein/fc_ecohydrology_scratch/CYGNSS/Data/sls_SOILMOISTURE_UVA/smap_sm_400m_2018_week_30.tif
-----------
[3] Skipping (already exists): /global/scratch/users/cgerlein/fc_ecohydrology_scratch/CYGNSS/Data/sls_SOILMOISTURE_UVA/smap_sm_400m_2018_week_31.tif
-----------
[4]Downloading: /global/scratch/users/cgerlein/fc_ecohydrology_scratch/CYGNSS/Data/sls_SOILMOISTURE_UVA/smap_sm_400m_2018_week_32.tif....
      Complete!
-----------
[5]Downloading: /global/scratch/users/cgerlein/fc_ecohydrology_scratch/CYGNSS/Data/sls_SOILMOISTURE_UVA/smap_sm_400m_2018_week_33.tif....
      Complete!
-----------
[6]Downloading: /global/

### Make a monthly composite

In [6]:
import rasterio
import numpy as np
import os
from datetime import datetime
import time

data_dir = "/global/scratch/users/cgerlein/fc_ecohydrology_scratch/CYGNSS/Data/sls_SOILMOISTURE_UVA/"
output_dir = f"{data_dir}monthly/"

def monthly_composite(month):
    print(f"[{datetime.now()}] Starting monthly composite for month {month}") ####
    start_total = time.time()
    
    # Get the weeks corresponding to the month...
    start_week = (month * 4) - 3
    end_week = month * 4
    
    # Get the files
    files = [
        f"{data_dir}smap_sm_400m_2015_week_{week:02d}.tif"
        for week in range(start_week, end_week + 1)
    ]

    arrays = []
    meta = None

    # Stack arrays
    i = 0
    for f in files:
        print(f"[{datetime.now()}] Reading week {i+1}/4") ####
        with rasterio.open(f) as src:
            arr = src.read(1)

            # REMOVE INVALID DATA
            arr[arr < 0] = np.nan
            arrays.append(arr)

            if meta is None:
                meta = src.meta.copy()
            i = i+1
    
    print(f"[{datetime.now()}] Stacking and Compositing") ####
    stack = np.stack(arrays)
    monthly = np.nanmean(stack, axis=0)

    # Save
    print(f"[{datetime.now()}] Saving") ####
    meta.update(dtype=rasterio.float32)
    output = f"{output_dir}smap_sm_400m_2015_month_{month:02d}.tif"
    with rasterio.open(output, "w", **meta) as dst:
        dst.write(monthly.astype(rasterio.float32), 1)
    end_total = time.time()
    print(f"[{datetime.now()}] Saved {output}, total {end_total - start_total:.2f} sec")
    return(output)

In [7]:
jan_ds = monthly_composite(1)

[2026-03-10 22:39:25.006742] Starting monthly composite for month 1
[2026-03-10 22:39:25.006799] Reading week 1/4
[2026-03-10 22:40:32.255436] Reading week 2/4


MemoryError: Unable to allocate 23.6 GiB for an array with shape (36540, 86760) and data type float64

In [ ]:
test = monthly_composite_dask(1)

[2026-03-10 22:46:13.869332] Starting monthly composite for month 1
[2026-03-10 22:46:13.869396] Reading week 1/4: /global/scratch/users/cgerlein/fc_ecohydrology_scratch/CYGNSS/Data/sls_SOILMOISTURE_UVA/smap_sm_400m_2015_week_01.tif (lazy)


#### Testing

In [14]:
# Let's open and check it out
import rioxarray
#ds = rioxarray.open_rasterio("/global/scratch/users/cgerlein/fc_ecohydrology_scratch/CYGNSS/Data/sls_SOILMOISTURE_UVA/smap_sm_400m_2015_week_04.tif")
ds[:,3000:4000, 3000:4000].max()

<xarray.DataArray ()> Size: 8B
array(nan)
Coordinates:
    spatial_ref  int64 8B 0
Attributes:
    AREA_OR_POINT:  Area
    _FillValue:     0.0
    scale_factor:   1.0
    add_offset:     0.0

In [5]:
ds['band'].min()

<xarray.DataArray 'band' ()> Size: 8B
array(1)
Coordinates:
    spatial_ref  int64 8B 0